In [1]:
import warnings
warnings.filterwarnings('ignore')

In [9]:
# check cuda gpu accelaration
import os
import torch
import numpy as np
import pandas as pd
import rasterio
import geopandas as gpd
import networkx as nx
from rasterio.windows import Window
from shapely.geometry import Polygon
from shapely.geometry import box
from PIL import Image
from rfdetr import RFDETRSegLarge
from pathlib import Path

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
np.set_printoptions(suppress=True)
device

device(type='cuda', index=0)

In [7]:
def calculate_accuracy_semantic_segmentation(groundtruth_gdf, prediction_shp):
    prediction_gdf = gpd.read_file(prediction_shp)
    intersection = gpd.overlay(groundtruth_gdf, prediction_gdf, how='intersection')
    intersection_area = intersection.area.sum()
    groundtruth_area = groundtruth_gdf.area.sum()
    prediction_area = prediction_gdf.area.sum()
    accuracy = intersection_area / (groundtruth_area + prediction_area - intersection_area) if (groundtruth_area + prediction_area - intersection_area) > 0 else 0
    return accuracy

def inference_large_tif_merge(model, tif_path, groundtruth_shp, output_shp, window_size=504, overlap=50, conf_thresh=0.4, merge_thresh=0.4):
    polygons = []
    confidences = []
    
    with rasterio.open(tif_path) as src:
        img_width, img_height = src.width, src.height
        tif_bounds = src.bounds
        crs = src.crs
        step = window_size - overlap

        x_starts = list(range(0, max(1, img_width - window_size + 1), step))
        if x_starts[-1] + window_size < img_width: x_starts.append(img_width - window_size)
        y_starts = list(range(0, max(1, img_height - window_size + 1), step))
        if y_starts[-1] + window_size < img_height: y_starts.append(img_height - window_size)

        total_windows = len(y_starts) * len(x_starts)
        count = 0

        for y in y_starts:
            for x in x_starts:
                count += 1
                window = Window(col_off=x, row_off=y, width=window_size, height=window_size)
                img_array = src.read([1, 2, 3], window=window)
                img_array = np.moveaxis(img_array, 0, -1)

                image_pil = Image.fromarray(img_array)
                results = model.predict(image_pil, threshold=conf_thresh)

                # Fix: Access masks and confidence from supervision Detections object
                if results.mask is not None:
                    win_transform = src.window_transform(window)
                    # supervision stores masks in .mask and confidence in .confidence
                    for i in range(len(results)):
                        mask = results.mask[i]
                        conf = results.confidence[i]

                        # Find contours to convert mask to polygon
                        import cv2
                        contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                        for contour in contours:
                            if len(contour) >= 3:
                                poly_pts = contour.reshape(-1, 2)
                                spatial_coords = [win_transform * (pt[0], pt[1]) for pt in poly_pts]
                                poly = Polygon(spatial_coords)
                                if poly.is_valid and poly.area > 0.1:
                                    polygons.append(poly)
                                    confidences.append(conf)

                if count % 50 == 0: print(f"Proses: {count}/{total_windows}")

    if not polygons:
        print("Tidak ada objek.")
        return

    gdf = gpd.GeoDataFrame({'geometry': polygons}, crs=crs)
    sindex = gdf.sindex
    G = nx.Graph()
    G.add_nodes_from(range(len(gdf)))

    for i in range(len(gdf)):
        poly_i = gdf.geometry.iloc[i]
        possible_matches = list(sindex.intersection(poly_i.bounds))
        for j in possible_matches:
            if i >= j: continue
            poly_j = gdf.geometry.iloc[j]
            try:
                inter_area = poly_i.intersection(poly_j).area
                if inter_area > 0:
                    ratio_i = inter_area / poly_i.area
                    ratio_j = inter_area / poly_j.area
                    if ratio_i > merge_thresh or ratio_j > merge_thresh:
                        G.add_edge(i, j)
            except: continue

    merged_polygons = []
    for component in nx.connected_components(G):
        group_geoms = [gdf.geometry.iloc[idx] for idx in component]
        merged_poly = gpd.GeoSeries(group_geoms).union_all()
        if merged_poly.is_valid:
            merged_polygons.append(merged_poly)

    final_gdf = gpd.GeoDataFrame({'id': range(1, len(merged_polygons) + 1), 'geometry': merged_polygons}, crs=crs)
    final_gdf.to_file(output_shp)

    groundtruth_gdf = gpd.read_file(groundtruth_shp)

    # Clip Groundtruh polygon with tif extent
    extent_geometry = box(*tif_bounds)
    gdf_extent = gpd.GeoDataFrame(geometry=[extent_geometry], crs=crs)
    groundtruth_gdf = gpd.clip(groundtruth_gdf, gdf_extent)

    accuracy = calculate_accuracy_semantic_segmentation(groundtruth_gdf, output_shp)
    print(f"\033[1mAkurasi {Path(tif_path).stem}: {accuracy}\033[0m")
    print(f"N groundtruth: {len(groundtruth_gdf)}; N Prediction: {len(merged_polygons)}")

    return Path(tif_path).stem, len(groundtruth_gdf), len(merged_polygons), accuracy * 100

In [ ]:
MODEL_PATH = "output/checkpoint_best_total.pth"
TIF_PATH = 'validation/Google/'
GROUNDTRUTH_PATH = 'validation/Groundtruth/'
OUTPUT_MERGED_PATH = 'validation/Prediction/'

df_val = pd.DataFrame(columns=["Val ID", "N Groundtruth", "N Prediction", "Accuracy IoU"])

model = RFDETRSegLarge(pretrain_weights=MODEL_PATH)
model.optimize_for_inference()

# create prediction folder if not exist
if not os.path.exists(OUTPUT_MERGED_PATH):
    os.mkdir(OUTPUT_MERGED_PATH)
    
# inference semua file.tif
for file in os.listdir(TIF_PATH):
    if file.endswith('.tif'):
        tif_path = os.path.join(TIF_PATH, file)
        groundtruth_shp = os.path.join(GROUNDTRUTH_PATH, os.path.splitext(file)[0], os.path.splitext(file)[0] + '.shp')
        if not os.path.exists(os.path.join(OUTPUT_MERGED_PATH, os.path.splitext(file)[0])):
            os.mkdir(os.path.join(OUTPUT_MERGED_PATH, os.path.splitext(file)[0]))
        output_shp = os.path.join(OUTPUT_MERGED_PATH, os.path.splitext(file)[0], os.path.splitext(file)[0] + '.shp')
        val_id, n_gt, n_p, accuracy = inference_large_tif_merge(model, tif_path, groundtruth_shp, output_shp, overlap=100, conf_thresh=0.4, merge_thresh=0.5)
        
        df_val.loc[len(df_val)] = [val_id, n_gt, n_p, accuracy]

[2026-07-09 13:17:56] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-07-09 13:17:56] [WARNING] rf-detr - Using patch size 12 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-07-09 13:18:32] [WARNING] rf-detr - Checkpoint has 1 classes but model is configured for 90. Using checkpoint class count (1). Pass num_classes=1 to suppress this warning.
[2026-07-09 13:18:32] [WARNING] rf-detr - load_pretrain_weights: args.num_queries absent; inferred ckpt_num_queries=200 from tensor rows 2600 ÷ ckpt_group_detr=13.


Akurasi val_16: 0.7547822405225145
N groundtruth: 92; N Prediction: 94
Akurasi val_3: 0.8953680798557371
N groundtruth: 49; N Prediction: 52
Akurasi val_8: 0.839168655269199
N groundtruth: 152; N Prediction: 151
Akurasi val_39: 0.8916630589313922
N groundtruth: 19; N Prediction: 16
Akurasi val_34: 0.921912150890893
N groundtruth: 42; N Prediction: 42
Akurasi val_44: 0.8605653863962659
N groundtruth: 338; N Prediction: 270
Akurasi val_15: 0.8173386294696597
N groundtruth: 102; N Prediction: 105
Akurasi val_4: 0.8292146651533047
N groundtruth: 26; N Prediction: 30
Akurasi val_36: 0.778674135905268
N groundtruth: 166; N Prediction: 174
Akurasi val_18: 0.7098828343749479
N groundtruth: 57; N Prediction: 64
Akurasi val_9a: 0.9420450636119048
N groundtruth: 83; N Prediction: 81
Akurasi val_43: 0.7795238592034711
N groundtruth: 92; N Prediction: 95
Akurasi val_33: 0.7826587661528072
N groundtruth: 91; N Prediction: 75
Akurasi val_20: 0.8159762252562431
N groundtruth: 207; N Prediction: 193
Ak

In [18]:
pd.set_option('display.float_format', lambda x: '%.2f' % x)
df_val

,Val ID,N Groundtruth,N Prediction,Accuracy IoU
0,val_16,92,94,75.48
1,val_3,49,52,89.54
2,val_8,152,151,83.92
3,val_39,19,16,89.17
4,val_34,42,42,92.19
5,val_44,338,270,86.06
6,val_15,102,105,81.73
7,val_4,26,30,82.92
8,val_36,166,174,77.87
9,val_18,57,64,70.99


In [17]:
df_val.to_excel("hasil_validasi.xlsx", sheet_name="Validation", index=False, float_format="%.2f")